## 1 - Dataset Acquisition and Initial Setup

In this first step, we are downloading the **FaceForensics++** dataset, the **c23** version, which corresponds to a moderate H.264 video compression (Constant Rate Factor 23). 

To streamline the acquisition process, we rely on the `kagglehub` library. This tool allows us to fetch the dataset directly from Kaggle and caches it locally.

In [1]:
import cv2
import glob
import kagglehub
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [2]:
print("Starting the download of the FaceForensics++ (c23) dataset...")
print("Note: This might take a while the first time...")

dataset_path = kagglehub.dataset_download("xdxd003/ff-c23")
print("Download completed or loaded from chache")
print("Dataset path:", dataset_path)

folder_items = os.listdir(dataset_path)
print("Contents inside", dataset_path, ": ")
print('\n'.join([f"- {item}" for item in folder_items]))

Starting the download of the FaceForensics++ (c23) dataset...
Note: This might take a while the first time...
Download completed or loaded from chache
Dataset path: /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1
Contents inside /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1 : 
- FaceForensics++_C23


In [3]:
base_path = os.path.join(dataset_path, "FaceForensics++_C23")
folders = os.listdir(base_path)
print("Contents inside", base_path, ": ")
print('\n'.join([f"- {f}" for f in folders]))

Contents inside /Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23 : 
- DeepFakeDetection
- Deepfakes
- original
- FaceShifter
- FaceSwap
- csv
- NeuralTextures
- Face2Face


In [4]:
for folder in folders:
    folder_path = os.path.join(base_path, folder)

    try:
        files = [file for file in os.listdir(folder_path)]

        if files:
            print(f"{folder}: {files[0]}")
        else:
            print(f"{folder}: Zero .mp4 files founded")

    except FileNotFoundError:
        print(f"{folder}: folder not founded")

DeepFakeDetection: 02_15__walking_and_outside_surprised__MZWH8ATN.mp4
Deepfakes: 475_265.mp4
original: 578.mp4
FaceShifter: 475_265.mp4
FaceSwap: 475_265.mp4
csv: NeuralTextures.csv
NeuralTextures: 475_265.mp4
Face2Face: 475_265.mp4


## 2 - Pandas DataFrame

This section creates a single pandas DataFrame containing all videos from the dataset.

Columns:
- `video`: filename
- `label`: 0 = real, 1 = fake
- `method`: manipulation type or "original"
- `target`: ID of the person being manipulated for fakes or identity of original
- `source`: ID of the source video for fakes or identity of original
- `sequence`: the specific name of the video sequence or scene recorded
- `experiment id`: an 8-character identifier used to distinguish multiple recordings of the same actor pairings
- `path`: full path to the video file

In [5]:
data = []

for folder in folders:
    folder_path = os.path.join(base_path, folder)

    if not os.path.exists(folder_path):
        continue

    if folder == "original":
        for video in os.listdir(folder_path):
            id = f"classic_{video.replace('.mp4', '')}"
            data.append({
                "video": video,
                "label": 0,
                "method": "original",
                "target": id,
                "source": id,
                "sequence": None,
                "exp_id": None,
                "path": os.path.join(folder_path, video)
            })

    elif folder == "DeepFakeDetection":
        for video in os.listdir(folder_path):
            if not video.endswith(".mp4"): continue

            name = video.replace(".mp4", "")
            parts = name.split("__")

            if len(parts) >= 3:
                actors = parts[0]
                sequence = parts[1]
                exp_id = parts[2]

                actors_parts = actors.split("_")
                if len(actors_parts) >= 2:
                    target = f"dfd_{actors_parts[0]}"
                    source = f"dfd_{actors_parts[1]}"
            else:
                continue

            data.append({
                "video": video,
                "label": 1,
                "method": folder,
                "target": target,
                "source": source,
                "sequence": sequence,
                "exp_id": exp_id,
                "path": os.path.join(folder_path, video)
            })

    else:
        for video in os.listdir(folder_path):
            if not video.endswith(".mp4"): continue

            name = video.replace(".mp4", "")
            parts = name.split("_")
            if len(parts) >= 2:
                target = f"classic_{parts[0]}"
                source = f"classic_{parts[1]}"
            else:
                continue

            data.append({
                "video": video,
                "label": 1,
                "method": folder,
                "target": target,
                "source": source,
                "sequence": None,
                "exp_id": None,
                "path": os.path.join(folder_path, video)
            })
    
videos = pd.DataFrame(data)

pd.set_option('display.max_colwidth', None)
display(videos.sample(15))

,video,label,method,target,source,sequence,exp_id,path
5141,131_518.mp4,1,NeuralTextures,classic_131,classic_518,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/131_518.mp4
758,06_25__outside_talking_pan_laughing__MI9BDQ7M.mp4,1,DeepFakeDetection,dfd_06,dfd_25,outside_talking_pan_laughing,MI9BDQ7M,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/06_25__outside_talking_pan_laughing__MI9BDQ7M.mp4
6952,271_264.mp4,1,Face2Face,classic_271,classic_264,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Face2Face/271_264.mp4
5788,709_390.mp4,1,NeuralTextures,classic_709,classic_390,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/709_390.mp4
160,26_12__podium_speech_happy__BFBNM8FR.mp4,1,DeepFakeDetection,dfd_26,dfd_12,podium_speech_happy,BFBNM8FR,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/26_12__podium_speech_happy__BFBNM8FR.mp4
6027,391_406.mp4,1,Face2Face,classic_391,classic_406,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Face2Face/391_406.mp4
147,17_16__hugging_happy__S7UMSIQV.mp4,1,DeepFakeDetection,dfd_17,dfd_16,hugging_happy,S7UMSIQV,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/17_16__hugging_happy__S7UMSIQV.mp4
5425,927_912.mp4,1,NeuralTextures,classic_927,classic_912,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/927_912.mp4
3216,976_954.mp4,1,FaceShifter,classic_976,classic_954,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/976_954.mp4
6793,209_016.mp4,1,Face2Face,classic_209,classic_016,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Face2Face/209_016.mp4


### 2.1 - Sanity Check

Quick checks to ensure the dataset is loaded correctly:

- Total number of videos
- Distribution of real vs fake videos
- Distribution of manipulation methods
- Number of unique target and source IDs

In [6]:
print("Total videos:", len(videos))
print("\nLabel distribution (fake/real):")
print(videos["label"].value_counts())
print("\nMethod distribution:")
print(videos["method"].value_counts())
print("\nTarget and Source identical for all 'original' videos:", (videos[videos['method'] == 'original']['target'] == videos[videos['method'] == 'original']['source']).all())
print("\nTarget unique number (identity):", videos["target"].nunique())
print("\nSource unique number:", videos["source"].nunique())

Total videos: 7000

Label distribution (fake/real):
label
1    6000
0    1000
Name: count, dtype: int64

Method distribution:
method
DeepFakeDetection    1000
Deepfakes            1000
original             1000
FaceShifter          1000
FaceSwap             1000
NeuralTextures       1000
Face2Face            1000
Name: count, dtype: int64

Target and Source identical for all 'original' videos: True

Target unique number (identity): 1028

Source unique number: 1028


### 2.2  - Distribution of Fake Videos per Target

Analyzes how many fake videos exist per target identity to identify if some identities dominate the fake samples

In [7]:
fake_df = videos[videos["label"] == 1]
per_target = fake_df.groupby("target").size()
print("\n--- FAKE PER TARGET STATISTICS ---")
print(per_target.describe())


--- FAKE PER TARGET STATISTICS ---
count    1028.000000
mean        5.836576
std         5.854495
min         5.000000
25%         5.000000
50%         5.000000
75%         5.000000
max        76.000000
dtype: float64


### 2.3 - Distribution Of Methods per Target (Check Variability):

Creates a pivot table showing how many videos of each manipulation method exist per target to verify that each identity has a representative set of manipulation methods and to detect identities with too few or missing manipulation types

In [8]:
pivot = pd.pivot_table(
    videos,
    index="target",
    columns="method",
    values="video",
    aggfunc="count",
    fill_value=0
)

print("\n--- METHOD DISTRIBUTION PER TARGET STATISTICS ---")
display(pivot.sample(20))


--- METHOD DISTRIBUTION PER TARGET STATISTICS ---


method,DeepFakeDetection,Deepfakes,Face2Face,FaceShifter,FaceSwap,NeuralTextures,original
target,,,,,,,
classic_317,0,1,1,1,1,1,1
classic_150,0,1,1,1,1,1,1
classic_056,0,1,1,1,1,1,1
classic_178,0,1,1,1,1,1,1
classic_733,0,1,1,1,1,1,1
classic_405,0,1,1,1,1,1,1
classic_810,0,1,1,1,1,1,1
classic_622,0,1,1,1,1,1,1
classic_290,0,1,1,1,1,1,1


### 2.4 - Check Label Balance

Checks the ratio of real to fake videos across the entire dataset to understand dataset imbalance

In [9]:
print("Normalize label ditribution:")
print(videos["label"].value_counts(normalize=True))

Normalize label ditribution:
label
1    0.857143
0    0.142857
Name: proportion, dtype: float64


### Dataset Observations

- **Total videos:** 7,000 in total, with 6,000 fake and 1,000 real videos. This presents a strong class imbalance (~6:1)
- **Target distribution is NOT uniform:** While the *median* is 5 fake videos per target, the dataset is heavily skewed by the `DeepFakeDetection` (DFD) subset. Because DFD relies on a small pool of professional actors filmed in multiple scenarios, some target identities appear in up to **76 fake videos**.
- **Manipulation methods:** The methods are perfectly balanced. There are exactly 1,000 videos for each of the 6 manipulation techniques (`Deepfakes`, `Face2Face`, `FaceSwap`, `NeuralTextures`, `FaceShifter`, `DeepFakeDetection`), plus the 1,000 `original` unmanipulated videos.
- **Target and source identities:** There are **2,028** unique target identities and **1,028** unique source identities. The "extra" 28 identities (beyond the base 1,000 subjects) belong to the hired actors from the Google DFD subset.

## 3 - Metadata Extraction (OpenCV)

In this cell, we use **OpenCV (`cv2`)** to iterate through our entire dataset of 7,000 videos and extract key metadata:
- `total_frames`
- `fps`
- `duration_sec`
- `resolution`

In [ ]:
tqdm.pandas(desc="Extracting video metadata")

videos_cv2 = videos.copy()
"""
Opens a video file momentarily just to read its metadata properties,
then closes it immediately to save memory.
"""
def get_video_metadata(video_path):
    try:
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return pd.Series([None, None, None, None])
    
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = total_frames / fps
        
        cap.release()

        return pd.Series([total_frames, fps, duration, f"{width}x{height}"])
    
    except Exception as e:
        return pd.Series([None, None, None, None])

print("Scanning all 7000 videos to extract physical metadata. This might take 1-3 minutes...")
videos[['total_frames', 'fps', 'duration_sec', 'resolution']] = videos['path'].progress_apply(get_video_metadata)

print("\nMetadata extraction complete!")

Scanning all 7000 videos to extract physical metadata. This might take 1-3 minutes...


Extracting video metadata: 100%|██████████| 7000/7000 [00:39<00:00, 176.59it/s]


Metadata extraction complete!


In [11]:
print("\n--- VIDEO DURATION STATISTICS (in seconds) ---")
print(videos['duration_sec'].describe())

print("\n--- VIDEO FRAMES STATISTICS ---")
print(videos['total_frames'].describe())

display(videos.sample(20))


--- VIDEO DURATION STATISTICS (in seconds) ---
count    7000.000000
mean       19.325502
std         9.536235
min         0.208333
25%        12.833333
50%        16.500000
75%        22.320000
max        72.560000
Name: duration_sec, dtype: float64

--- VIDEO FRAMES STATISTICS ---
count    7000.000000
mean      511.797143
std       229.865219
min         5.000000
25%       346.000000
50%       444.000000
75%       594.000000
max      1814.000000
Name: total_frames, dtype: float64


,video,label,method,target,source,sequence,exp_id,path,total_frames,fps,duration_sec,resolution
997,19_24__outside_talking_still_laughing__DL7X7M6I.mp4,1,DeepFakeDetection,dfd_19,dfd_24,outside_talking_still_laughing,DL7X7M6I,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/19_24__outside_talking_still_laughing__DL7X7M6I.mp4,837,24.0,34.875000,1920x1080
3328,626_562.mp4,1,FaceShifter,classic_626,classic_562,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/FaceShifter/626_562.mp4,975,30.0,32.500000,854x480
1692,043_110.mp4,1,Deepfakes,classic_043,classic_110,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/Deepfakes/043_110.mp4,495,30.0,16.500000,1280x720
865,18_12__outside_talking_pan_laughing__IKH1LBBY.mp4,1,DeepFakeDetection,dfd_18,dfd_12,outside_talking_pan_laughing,IKH1LBBY,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/18_12__outside_talking_pan_laughing__IKH1LBBY.mp4,683,24.0,28.458333,1920x1080
765,27_18__outside_talking_pan_laughing__NYHE8D0J.mp4,1,DeepFakeDetection,dfd_27,dfd_18,outside_talking_pan_laughing,NYHE8D0J,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/27_18__outside_talking_pan_laughing__NYHE8D0J.mp4,668,24.0,27.833333,1920x1080
5027,391_406.mp4,1,NeuralTextures,classic_391,classic_406,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/NeuralTextures/391_406.mp4,512,25.0,20.480000,640x480
2621,317.mp4,0,original,classic_317,classic_317,None,None,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/original/317.mp4,682,30.0,22.733333,640x480
385,20_25__exit_phone_room__M86GQTHK.mp4,1,DeepFakeDetection,dfd_20,dfd_25,exit_phone_room,M86GQTHK,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/20_25__exit_phone_room__M86GQTHK.mp4,574,24.0,23.916667,1920x1080
840,11_21__kitchen_still__N6YQW2AP.mp4,1,DeepFakeDetection,dfd_11,dfd_21,kitchen_still,N6YQW2AP,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/11_21__kitchen_still__N6YQW2AP.mp4,774,24.0,32.250000,1920x1080
140,06_20__kitchen_pan__6SUW7063.mp4,1,DeepFakeDetection,dfd_06,dfd_20,kitchen_pan,6SUW7063,/Users/luciapola/.cache/kagglehub/datasets/xdxd003/ff-c23/versions/1/FaceForensics++_C23/DeepFakeDetection/06_20__kitchen_pan__6SUW7063.mp4,686,24.0,28.583333,1920x1080


In [12]:
print("Creation of the videos copy and extraction of extended data from the CSV…")

videos_csv = videos.copy()

metadata_csv_path = os.path.join(base_path, "csv", "FF++_Metadata.csv")
ff_metadata = pd.read_csv(metadata_csv_path)

ff_metadata['csv_method'] = ff_metadata['File Path'].apply(lambda x: Path(x).parts[0])
ff_metadata['csv_video'] = ff_metadata['File Path'].apply(lambda x: Path(x).name)
ff_metadata = ff_metadata.drop_duplicates(subset=['csv_method', 'csv_video'])

ff_metadata['csv_res_string'] = ff_metadata['Width'].astype(str) + "x" + ff_metadata['Height'].astype(str)

videos_csv = videos_csv.merge(
    ff_metadata[['csv_method', 'csv_video', 'Frame Count', 'csv_res_string']], 
    left_on=['method', 'video'], 
    right_on=['csv_method', 'csv_video'], 
    how='left'
)

videos_csv = videos_csv.rename(columns={
    'Frame Count': 'csv_frames',
    'csv_res_string': 'csv_resolution'
})
videos_csv = videos_csv.drop(columns=['csv_method', 'csv_video'])

print("\nCSV extraction complete!")
print(f"Rows: {len(videos_csv)}")

Creation of the videos copy and extraction of extended data from the CSV…

CSV extraction complete!
Rows: 7000


## 5 - Data Integrity Verification (OpenCV vs. CSV)

To ensure absolute data integrity, we cross-reference the video properties (frame count and resolution) dynamically extracted via OpenCV with the official metadata provided by the dataset authors.

In [13]:
print("--- VERIFICATION: OpenCV vs. CSV ---")

#Comparison DataFrame
comparison_df = pd.DataFrame({
    'video': videos['video'],
    'method': videos['method'],
    'opencv_frames': videos['total_frames'],
    'csv_frames': videos_csv['csv_frames'],
    'opencv_resolution': videos['resolution'],
    'csv_resolution': videos_csv['csv_resolution']
})

mask_frames = comparison_df['opencv_frames'] != comparison_df['csv_frames']
mask_res = comparison_df['opencv_resolution'] != comparison_df['csv_resolution']

mismatched_videos = comparison_df[(mask_frames | mask_res)].dropna()

if mismatched_videos.empty:
    print("OK: Frame counts and Resolution (Width x Height) match perfectly across all 7000 videos.")
else:
    print(f"WARNING: Found discrepancies in {len(mismatched_videos)} videos.")
    display(mismatched_videos.sample(10))

--- VERIFICATION: OpenCV vs. CSV ---
OK: Frame counts and Resolution (Width x Height) match perfectly across all 7000 videos.


### Metadata Observations

The comparison reveals that the frame counts match perfectly between OpenCV and the CSV, confirming that the physical video files are intact and uncorrupted.

## 6 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_videos/dataframe_videos`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [14]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "fplusplus_videos.csv")
videos.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")

--- SAVING DATAFRAME ---
Data succesfully saved to: ./processed_images/fplusplus_videos.csv
